In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# تحميل ملف التدريب لأسعار المنازل (تأكد من وضع مسار الملف الصحيح لديك)
df_train = pd.read_csv('train.csv')

# استعراض الأبعاد وأول 5 صفوف
print("--- Dataset Shape ---")
print(df_train.shape)

print("\n--- First 5 Rows ---")
display(df_train.head())

print("\n--- Data Info (Data Types & Non-Null Counts) ---")
df_train.info()

--- Dataset Shape ---
(1460, 81)

--- First 5 Rows ---


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000



--- Data Info (Data Types & Non-Null Counts) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  Ov

In [4]:
# 1. تحميل الداتا من جديد والتأكد من حذف الId
df_train = pd.read_csv('train.csv')
df_train.drop('Id', axis=1, inplace=True)

# 2. معالجة كافة القيم المفقودة للأعمدة النصية (تعويضها بـ 'None')
cat_cols_with_na = ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
                    'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType',
                    'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']
for col in cat_cols_with_na:
    df_train[col] = df_train[col].fillna('None')

# 3. معالجة القيم المفقودة للأعمدة العددية الأخرى (تعويضها بـ 0 أو Median)
df_train['MasVnrArea'] = df_train['MasVnrArea'].fillna(0)
df_train['GarageYrBlt'] = df_train['GarageYrBlt'].fillna(0)
df_train['LotFrontage'] = df_train['LotFrontage'].fillna(df_train['LotFrontage'].median())

# بالنسبة لأي عمود نصي آخر قد تبقى فيه قيمة مفقودة بسيطة، نعوضها بـ 'Missing'
for col in df_train.select_dtypes(include=['object']).columns:
    df_train[col] = df_train[col].fillna('Missing')

# التأكد من عدم وجود أي قيمة مفقودة نهائياً
print("Total Remaining Missing Values:", df_train.isnull().sum().sum())

# 4. تحويل الأعمدة الرقمية التي تمثل فئات (Categories) إلى نصية لكي يتم ترميزها بشكل صحيح
df_train['MSSubClass'] = df_train['MSSubClass'].astype(str)
df_train['MoSold'] = df_train['MoSold'].astype(str)

print("Data cleaning and types adjustment completed successfully!")
print("Current Shape:", df_train.shape)

Total Remaining Missing Values: 0
Data cleaning and types adjustment completed successfully!
Current Shape: (1460, 80)


In [5]:
# 1. تحويل كافة الأعمدة النصية (Categorical) إلى أعمدة رقمية باستخدام One-Hot Encoding
df_train_encoded = pd.get_dummies(df_train, drop_first=True, dtype=int)

# 2. فصل المتغيرات المستقلة (Features - X) عن المتغير المستهدف (Target - y -> SalePrice)
X = df_train_encoded.drop('SalePrice', axis=1)
y = df_train_encoded['SalePrice']

print("Shape of X (Features):", X.shape)
print("Shape of y (Target):", y.shape)

# 3. تقسيم البيانات إلى مجموعة تدريب (Train) ومجموعة اختبار (Test) بنسبة 80% لـ 20%
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n--- Train-Test Split Completed Successfully! ---")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Shape of X (Features): (1460, 283)
Shape of y (Target): (1460,)

--- Train-Test Split Completed Successfully! ---
X_train shape: (1168, 283)
X_test shape: (292, 283)


In [6]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. تعريف موديلات الانحدار
models = {
    "Ridge Regression": Ridge(alpha=10.0, random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "XGBoost Regressor": XGBRegressor(random_state=42, eval_metric='rmse')
}

print("--- Regression Models Evaluation Results ---\n")

# 2. حلقة تدريب وتقييم كل موديل
for name, model in models.items():
    # تدريب النموذج
    model.fit(X_train, y_train)

    # التنبؤ على بيانات الاختبار
    y_pred = model.predict(X_test)

    # حساب مقاييس الأداء: R2 Score (معامل التحديد) و RMSE (جذر متوسط مربع الخطأ)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"Model: {name}")
    print(f"R2 Score: {r2:.4f} (Accuracy equivalent)")
    print(f"RMSE (Root Mean Squared Error): {rmse:.2f}")
    print("-" * 45)

--- Regression Models Evaluation Results ---

Model: Ridge Regression
R2 Score: 0.8767 (Accuracy equivalent)
RMSE (Root Mean Squared Error): 30755.89
---------------------------------------------
Model: Random Forest Regressor
R2 Score: 0.8896 (Accuracy equivalent)
RMSE (Root Mean Squared Error): 29096.14
---------------------------------------------
Model: XGBoost Regressor
R2 Score: 0.8947 (Accuracy equivalent)
RMSE (Root Mean Squared Error): 28420.72
---------------------------------------------


In [7]:
from sklearn.metrics import r2_score, mean_squared_error

print("--- Overfitting Check (Train vs Test Performance) ---\n")

for name, model in models.items():
    # التنبؤ على بيانات التدريب وببيانات الاختبار
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # حساب R2 للطرفين
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"Model: {name}")
    print(f"   Train R2 Score: {train_r2:.4f}")
    print(f"   Test R2 Score:  {test_r2:.4f}")
    print(f"   Difference:     {(train_r2 - test_r2):.4f}")
    print("-" * 45)

--- Overfitting Check (Train vs Test Performance) ---

Model: Ridge Regression
   Train R2 Score: 0.8966
   Test R2 Score:  0.8767
   Difference:     0.0199
---------------------------------------------
Model: Random Forest Regressor
   Train R2 Score: 0.9788
   Test R2 Score:  0.8896
   Difference:     0.0892
---------------------------------------------
Model: XGBoost Regressor
   Train R2 Score: 0.9997
   Test R2 Score:  0.8947
   Difference:     0.1050
---------------------------------------------


In [8]:
# تعديل معلمات النماذج لتقليل الـ Overfitting
models_tuned = {
    "Ridge Regression (Tuned)": Ridge(alpha=20.0, random_state=42),
    # تقييد عمق الأشجار ومنع الحفظ الزائد
    "Random Forest (Tuned)": RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=5, random_state=42),
    # تقليل معدل التعلم وزيادة التنظيم في XGBoost
    "XGBoost (Tuned)": XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42, eval_metric='rmse')
}

print("--- Tuned Models Overfitting Check ---\n")

for name, model in models_tuned.items():
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"Model: {name}")
    print(f"   Train R2 Score: {train_r2:.4f}")
    print(f"   Test R2 Score:  {test_r2:.4f}")
    print(f"   Difference:     {(train_r2 - test_r2):.4f}")
    print("-" * 45)

--- Tuned Models Overfitting Check ---

Model: Ridge Regression (Tuned)
   Train R2 Score: 0.8878
   Test R2 Score:  0.8747
   Difference:     0.0130
---------------------------------------------
Model: Random Forest (Tuned)
   Train R2 Score: 0.9725
   Test R2 Score:  0.8900
   Difference:     0.0825
---------------------------------------------
Model: XGBoost (Tuned)
   Train R2 Score: 0.9800
   Test R2 Score:  0.9227
   Difference:     0.0574
---------------------------------------------
